# Chapter 4.6: Speculative Decoding in vLLM

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** the draft-verify paradigm and why speculative decoding accelerates inference
2. **Derive** the acceptance/rejection sampling mathematics
3. **Compare** draft model, Medusa, Eagle, and MLPSpeculator approaches
4. **Trace** through vLLM's speculative execution source code
5. **Analyze** when speculation helps and when it hurts
6. **Measure** acceptance rates and throughput improvements
7. **Configure** speculative decoding in vLLM

---

## Prerequisites
- Understanding of autoregressive LLM decoding
- Basic probability theory
- Chapter 4.3 (CUDA Graphs — relevant for decode optimization)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part4/chapter_4.6_speculative_decoding.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part4/chapter_4.6_speculative_decoding.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The Decode Bottleneck

### Why Autoregressive Decoding is Slow

In standard autoregressive decoding, each token is generated sequentially:

```
Step 1: [prompt] → model → token_1
Step 2: [prompt, token_1] → model → token_2
Step 3: [prompt, token_1, token_2] → model → token_3
...

Each step requires a FULL forward pass through the model.
For large models, each step is memory-bandwidth bound:
  - Must read ALL model weights from HBM
  - But only computes for 1 token
  - GPU compute utilization: < 1%

Arithmetic intensity (FLOPs/byte):
  Prefill (many tokens): ~100 → compute bound
  Decode (1 token):      ~0.5 → memory bound
```

### Speculative Decoding Insight

**Key idea**: Use a cheap draft model to guess multiple tokens, then verify them in parallel with the target model.

```
Standard Decoding (5 tokens = 5 forward passes):
  [Target FWD] → t1
  [Target FWD] → t2
  [Target FWD] → t3
  [Target FWD] → t4
  [Target FWD] → t5

Speculative Decoding (5 tokens = 1 draft + 1 verify):
  [Draft FWD ×5] → d1, d2, d3, d4, d5  (fast, small model)
  [Target FWD ×1] → verify all 5 simultaneously
  Accept: d1 ✓, d2 ✓, d3 ✓, d4 ✗ → keep 3 tokens + sample 1 new
  Result: 4 tokens from 1 target pass!
```

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Dict
import time
import random
from collections import Counter

torch.manual_seed(42)
np.random.seed(42)

print("Imports loaded.")

## 2. Acceptance/Rejection Sampling Mathematics

### The Core Algorithm (Leviathan et al., 2022; Chen et al., 2023)

Given:
- Target model distribution: $p(x_t | x_{<t})$
- Draft model distribution: $q(x_t | x_{<t})$
- Draft token: $\tilde{x}_t \sim q(x_t | x_{<t})$

**Acceptance criterion**: Accept $\tilde{x}_t$ with probability:
$$\alpha = \min\left(1, \frac{p(\tilde{x}_t)}{q(\tilde{x}_t)}\right)$$

If rejected, sample from the **residual distribution**:
$$p'(x) = \frac{\max(0, p(x) - q(x))}{\sum_{x'} \max(0, p(x') - q(x'))}$$

### Why This Works (Proof Sketch)

The combined accept/reject process samples from the **exact** target distribution $p$:

$$P(\text{output} = x) = q(x) \cdot \min\left(1, \frac{p(x)}{q(x)}\right) + (1 - \alpha) \cdot p'(x) = p(x)$$

This is mathematically exact — speculative decoding produces the same distribution as standard decoding!

In [ ]:
def speculative_sampling(
    target_probs: torch.Tensor,  # [vocab_size] - p(x)
    draft_probs: torch.Tensor,   # [vocab_size] - q(x)
    draft_token: int,             # Sampled from q
) -> Tuple[int, bool]:
    """
    Speculative sampling: accept/reject a draft token.
    
    Returns: (accepted_token, was_accepted)
    
    This mirrors the logic in vllm/spec_decode/spec_decode_worker.py
    """
    # Acceptance probability
    p = target_probs[draft_token].item()
    q = draft_probs[draft_token].item()
    
    acceptance_prob = min(1.0, p / max(q, 1e-10))
    
    # Accept or reject
    u = random.random()
    if u < acceptance_prob:
        return draft_token, True
    else:
        # Sample from residual distribution
        residual = torch.clamp(target_probs - draft_probs, min=0)
        residual = residual / residual.sum()  # Normalize
        resampled_token = torch.multinomial(residual, 1).item()
        return resampled_token, False


def verify_speculative_correctness(num_samples: int = 100000):
    """
    Empirically verify that speculative sampling produces
    the exact target distribution.
    """
    vocab_size = 10
    
    # Create target and draft distributions
    target_logits = torch.randn(vocab_size)
    draft_logits = target_logits + torch.randn(vocab_size) * 0.5  # Noisy approximation
    
    target_probs = F.softmax(target_logits, dim=0)
    draft_probs = F.softmax(draft_logits, dim=0)
    
    # Sample from target directly
    direct_samples = torch.multinomial(target_probs, num_samples, replacement=True)
    
    # Sample via speculative decoding
    spec_samples = []
    accepts = 0
    for _ in range(num_samples):
        # Draft: sample from q
        draft_token = torch.multinomial(draft_probs, 1).item()
        # Verify
        token, accepted = speculative_sampling(target_probs, draft_probs, draft_token)
        spec_samples.append(token)
        if accepted:
            accepts += 1
    
    # Compare distributions
    direct_counts = Counter(direct_samples.tolist())
    spec_counts = Counter(spec_samples)
    
    print("Verification: Speculative Sampling produces target distribution")
    print(f"{'Token':>6} {'Target p':>10} {'Direct %':>10} {'Spec %':>10} {'Diff':>8}")
    print("-" * 48)
    
    max_diff = 0
    for token in range(vocab_size):
        p_true = target_probs[token].item()
        p_direct = direct_counts.get(token, 0) / num_samples
        p_spec = spec_counts.get(token, 0) / num_samples
        diff = abs(p_direct - p_spec)
        max_diff = max(max_diff, diff)
        print(f"{token:>6} {p_true:>10.4f} {p_direct:>10.4f} {p_spec:>10.4f} {diff:>8.4f}")
    
    print(f"\nAcceptance rate: {accepts/num_samples:.1%}")
    print(f"Max distribution difference: {max_diff:.4f}")
    print(f"Distributions match: {max_diff < 0.02}")

verify_speculative_correctness()

## 3. Expected Acceptance Rate Analysis

### Theoretical Acceptance Rate

The expected acceptance rate $\alpha$ depends on how similar the draft and target distributions are:

$$E[\alpha] = \sum_x q(x) \min\left(1, \frac{p(x)}{q(x)}\right) = \sum_x \min(p(x), q(x)) = 1 - \frac{1}{2}\|p - q\|_1$$

where $\|p - q\|_1$ is the total variation distance.

### Expected Tokens per Verification Step

If we generate $K$ draft tokens, the expected number of accepted tokens is:
$$E[\text{accepted}] = \frac{1 - \alpha^{K+1}}{1 - \alpha}$$

(Plus one token from the residual distribution if any draft is rejected)

In [ ]:
def expected_tokens_per_step(alpha: float, K: int) -> float:
    """
    Expected number of tokens accepted per verification step.
    
    K: number of draft tokens
    alpha: per-token acceptance rate
    """
    if alpha >= 1.0:
        return K + 1
    if alpha <= 0:
        return 1
    
    # Expected accepted tokens = sum_{i=0}^{K} alpha^i * (1 - alpha) + alpha^{K+1}
    # = (1 - alpha^{K+1}) / (1 - alpha) + alpha^{K+1}
    # Simplified: always get at least 1 (from residual or last accept)
    return (1 - alpha**(K+1)) / (1 - alpha)


def speculative_speedup(
    alpha: float,
    K: int,
    draft_cost: float = 0.1,  # Cost of draft model relative to target
) -> float:
    """
    Calculate speedup from speculative decoding.
    
    Standard: 1 target forward per token
    Speculative: K draft forwards + 1 target forward per step
    """
    expected_tokens = expected_tokens_per_step(alpha, K)
    
    # Cost per step (relative)
    spec_cost = K * draft_cost + 1  # K draft + 1 target
    standard_cost = expected_tokens  # Same number of target forwards
    
    return standard_cost / spec_cost


# Analyze speedup for various acceptance rates and K values
alphas = np.linspace(0.1, 0.99, 50)
K_values = [1, 3, 5, 7, 10]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Expected tokens per step
for K in K_values:
    tokens = [expected_tokens_per_step(a, K) for a in alphas]
    ax1.plot(alphas, tokens, label=f'K={K}', linewidth=2)

ax1.set_xlabel('Acceptance Rate (alpha)', fontsize=12)
ax1.set_ylabel('Expected Tokens per Step', fontsize=12)
ax1.set_title('Expected Tokens Accepted per Verification Step', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Speedup
for K in K_values:
    speedups = [speculative_speedup(a, K, draft_cost=0.05) for a in alphas]
    ax2.plot(alphas, speedups, label=f'K={K}', linewidth=2)

ax2.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='No speedup')
ax2.set_xlabel('Acceptance Rate (alpha)', fontsize=12)
ax2.set_ylabel('Speedup', fontsize=12)
ax2.set_title('Speculative Decoding Speedup (draft cost=5%)', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/speculative_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Draft Model Approach

### Architecture

```
Draft Model Approach:

┌──────────────────────────────────────────────────┐
│ Step 1: DRAFT (fast, small model)                 │
│                                                    │
│ Draft Model (e.g., 68M params)                    │
│ [context] → d1 → d2 → d3 → d4 → d5              │
│                                                    │
│ Time: 5 × T_draft (very fast)                     │
└──────────────────────────────────────────────────┘
                    │
                    ▼
┌──────────────────────────────────────────────────┐
│ Step 2: VERIFY (large target model, single pass)  │
│                                                    │
│ Target Model (e.g., 70B params)                   │
│ [context, d1, d2, d3, d4, d5] → p1, p2, p3, p4, p5│
│                                                    │
│ Time: 1 × T_target (single forward pass!)         │
└──────────────────────────────────────────────────┘
                    │
                    ▼
┌──────────────────────────────────────────────────┐
│ Step 3: ACCEPT/REJECT                             │
│                                                    │
│ For each draft token di:                          │
│   If p(di) / q(di) > uniform(0,1): ACCEPT        │
│   Else: REJECT, sample from residual, STOP       │
│                                                    │
│ Result: Accept first k tokens + 1 from residual   │
└──────────────────────────────────────────────────┘
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Figure 1: Speculative Decoding Flow ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16); ax.set_ylim(0, 8); ax.axis('off')
fig.patch.set_facecolor('white')

# Step 1: Draft model generates K tokens
ax.add_patch(mpatches.FancyBboxPatch((1, 6.5), 4.5, 1.2, boxstyle="round,pad=0.15",
    facecolor=ORANGE, alpha=0.9, edgecolor='white', linewidth=2))
ax.text(3.25, 7.1, 'DRAFT MODEL (small, fast)\nGenerate K=5 tokens: d1 d2 d3 d4 d5',
        ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# Arrow
ax.annotate('', xy=(8, 7.1), xytext=(5.5, 7.1),
            arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2.5))
ax.text(6.75, 7.5, 'All K tokens\nat once', fontsize=8, ha='center', color='#555')

# Step 2: Target model verifies
ax.add_patch(mpatches.FancyBboxPatch((8, 6.5), 6.5, 1.2, boxstyle="round,pad=0.15",
    facecolor=BLUE, alpha=0.9, edgecolor='white', linewidth=2))
ax.text(11.25, 7.1, 'TARGET MODEL (large)\nSingle forward pass: verify d1 d2 d3 d4 d5 simultaneously',
        ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# Arrow down
ax.annotate('', xy=(8, 5.2), xytext=(8, 6.5),
            arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2.5))

# Step 3: Accept/Reject
ax.add_patch(mpatches.FancyBboxPatch((2, 3.5), 12, 2.0, boxstyle="round,pad=0.15",
    facecolor='#FDEBD0', edgecolor=ORANGE, linewidth=2))
ax.text(8, 5.0, 'Accept / Reject Each Draft Token', fontsize=12,
        ha='center', fontweight='bold', color=ORANGE)

# Token-by-token verdict
tokens = [
    ('d1', GREEN, 'ACCEPT'),
    ('d2', GREEN, 'ACCEPT'),
    ('d3', GREEN, 'ACCEPT'),
    ('d4', RED,   'REJECT'),
    ('d5', '#BDC3C7', 'skip'),
]
for i, (tok, color, verdict) in enumerate(tokens):
    x = 3 + i * 2.2
    rect = mpatches.FancyBboxPatch((x, 3.8), 1.8, 1.2, boxstyle="round,pad=0.08",
        facecolor=color, alpha=0.8, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+0.9, 4.7, tok, ha='center', fontsize=10, fontweight='bold', color='white')
    ax.text(x+0.9, 4.1, verdict, ha='center', fontsize=8, color='white', fontweight='bold')

# Result
ax.add_patch(mpatches.FancyBboxPatch((2, 1.5), 12, 1.5, boxstyle="round,pad=0.15",
    facecolor=GREEN, alpha=0.85, edgecolor='white', linewidth=2))
ax.text(8, 2.55, 'Result: 3 accepted + 1 resampled = 4 tokens from 1 target forward pass!',
        ha='center', fontsize=11, fontweight='bold', color='white')
ax.text(8, 1.9, 'Standard decoding: 4 tokens = 4 target forward passes. Speedup: up to 4x.',
        ha='center', fontsize=9, color='white', style='italic')

ax.annotate('', xy=(8, 3.0), xytext=(8, 3.5),
            arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

ax.set_title('Speculative Decoding: Draft-then-Verify Flow',
             fontsize=15, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

In [ ]:
class DraftModelSimulator:
    """
    Simulates speculative decoding with a draft model.
    
    Source reference: vllm/spec_decode/spec_decode_worker.py
    """
    
    def __init__(self, vocab_size: int = 1000, 
                 target_quality: float = 1.0,
                 draft_quality: float = 0.7,
                 K: int = 5):
        self.vocab_size = vocab_size
        self.target_quality = target_quality
        self.draft_quality = draft_quality
        self.K = K
        
        # Metrics
        self.total_steps = 0
        self.total_tokens = 0
        self.total_accepts = 0
        self.total_draft_tokens = 0
        self.acceptance_history = []
    
    def _generate_distributions(self):
        """
        Generate target and draft probability distributions.
        The draft approximates the target with some noise.
        """
        # Target distribution (sharp)
        target_logits = torch.randn(self.vocab_size) * self.target_quality
        target_probs = F.softmax(target_logits * 2, dim=0)  # Temperature for sharpness
        
        # Draft: noisy version of target
        noise = torch.randn(self.vocab_size) * (1 - self.draft_quality)
        draft_logits = target_logits * self.draft_quality + noise
        draft_probs = F.softmax(draft_logits * 2, dim=0)
        
        return target_probs, draft_probs
    
    def step(self) -> dict:
        """
        Execute one speculative decoding step.
        Returns: dict with step metrics.
        """
        self.total_steps += 1
        accepted_tokens = 0
        
        # Generate K draft tokens
        for k in range(self.K):
            target_probs, draft_probs = self._generate_distributions()
            
            # Sample draft token
            draft_token = torch.multinomial(draft_probs, 1).item()
            self.total_draft_tokens += 1
            
            # Accept/reject
            _, accepted = speculative_sampling(target_probs, draft_probs, draft_token)
            
            if accepted:
                accepted_tokens += 1
                self.total_accepts += 1
            else:
                # Rejected: we still get 1 token from residual
                accepted_tokens += 1  # The resampled token
                break
        else:
            # All K accepted: bonus token from target
            accepted_tokens += 1
        
        self.total_tokens += accepted_tokens
        self.acceptance_history.append(accepted_tokens)
        
        return {
            'accepted_tokens': accepted_tokens,
            'draft_tokens': min(self.K, accepted_tokens),
        }
    
    @property
    def avg_tokens_per_step(self):
        return self.total_tokens / max(self.total_steps, 1)
    
    @property
    def acceptance_rate(self):
        return self.total_accepts / max(self.total_draft_tokens, 1)

# Simulate
print(f"{'Quality':>8} {'K':>3} {'Accept Rate':>12} {'Avg Tok/Step':>13} {'Speedup':>8}")
print("-" * 50)

for quality in [0.5, 0.7, 0.85, 0.95]:
    for K in [3, 5, 7]:
        sim = DraftModelSimulator(draft_quality=quality, K=K)
        for _ in range(1000):
            sim.step()
        
        speedup = sim.avg_tokens_per_step / (K * 0.05 + 1)  # draft_cost=5%
        print(f"{quality:>8.2f} {K:>3} {sim.acceptance_rate:>11.1%} "
              f"{sim.avg_tokens_per_step:>12.2f} {speedup:>7.2f}x")

## 5. Medusa: Multi-Head Speculative Decoding

### Key Idea

Instead of a separate draft model, Medusa adds **extra prediction heads** to the target model:

```
Standard LLM:
  hidden_state → LM_head → next_token

Medusa LLM:
  hidden_state → LM_head   → next_token (t+1)
               → Medusa_1  → token (t+2)
               → Medusa_2  → token (t+3)
               → Medusa_3  → token (t+4)
               → Medusa_4  → token (t+5)

Each Medusa head predicts a future token position.
No separate draft model needed!
```

### Advantages
- No separate draft model to manage
- Shares the target model's hidden states (high quality)
- Medusa heads are tiny (single linear layer each)
- Easy to train on top of existing models

In [ ]:
class MedusaHead(torch.nn.Module):
    """
    A Medusa prediction head.
    
    Each head predicts a token at a specific future position.
    Implemented as a simple MLP with residual connection.
    """
    
    def __init__(self, hidden_size: int, vocab_size: int):
        super().__init__()
        self.linear1 = torch.nn.Linear(hidden_size, hidden_size)
        self.act = torch.nn.SiLU()
        self.linear2 = torch.nn.Linear(hidden_size, vocab_size)
    
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """Predict logits for a future position."""
        h = self.act(self.linear1(hidden_states))
        return self.linear2(h + hidden_states)  # Residual


class MedusaModel:
    """
    Simulates a Medusa-augmented model.
    """
    
    def __init__(self, hidden_size: int = 256, vocab_size: int = 1000, num_heads: int = 4):
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
        self.num_heads = num_heads
        
        # Original LM head
        self.lm_head = torch.nn.Linear(hidden_size, vocab_size)
        
        # Medusa heads
        self.medusa_heads = torch.nn.ModuleList([
            MedusaHead(hidden_size, vocab_size)
            for _ in range(num_heads)
        ])
    
    def forward(self, hidden_states: torch.Tensor):
        """
        Returns predictions for current + future positions.
        """
        # Standard prediction (position t+1)
        logits_0 = self.lm_head(hidden_states)
        
        # Medusa predictions (positions t+2, t+3, ...)
        medusa_logits = [head(hidden_states) for head in self.medusa_heads]
        
        return logits_0, medusa_logits

# Demo
model = MedusaModel(hidden_size=256, vocab_size=1000, num_heads=4)

# Count parameters
total_params = sum(p.numel() for p in model.lm_head.parameters())
medusa_params = sum(p.numel() for head in model.medusa_heads for p in head.parameters())

print(f"LM head parameters: {total_params:,}")
print(f"Medusa heads parameters: {medusa_params:,} ({model.num_heads} heads)")
print(f"Medusa overhead: {medusa_params/total_params*100:.1f}% of LM head")

# Forward pass
hidden = torch.randn(1, 256)
logits_0, medusa_logits = model.forward(hidden)
print(f"\nPredictions: position t+1 logits shape: {logits_0.shape}")
for i, ml in enumerate(medusa_logits):
    print(f"  Medusa head {i}: position t+{i+2} logits shape: {ml.shape}")

## 6. Eagle: Feature-Level Draft Model

### Key Innovation

Eagle operates at the **feature level** (hidden states) rather than the token level:

```
Standard Draft Model:
  draft_model(tokens) → draft_logits → draft_tokens
  Limitation: must encode tokens back to features

Eagle:
  target_model.hidden_states → eagle_head → draft_features
  draft_features → target_model.lm_head → draft_logits → draft_tokens
  
  Advantage: 
  - Reuses target model's feature space directly
  - Higher acceptance rates
  - Autoregressive at feature level (not token level)
```

### Architecture

```
Target model hidden state (from last layer)
    │
    ▼
┌──────────────────────┐
│ Eagle Feature Head    │
│ (lightweight decoder) │
│                      │
│ Input: h_t (features)│
│ Output: h_{t+1} pred │
│         h_{t+2} pred │
│         h_{t+3} pred │
└──────────────────────┘
    │
    ▼
Target model's LM head (shared)
    │
    ▼
Draft token predictions
```

In [ ]:
# Compare acceptance rates of different speculative methods

def simulate_acceptance_rates(method: str, num_trials: int = 5000) -> List[float]:
    """
    Simulate acceptance rates for different speculative decoding methods.
    """
    rates = []
    vocab_size = 1000
    
    for _ in range(num_trials):
        # Target distribution
        target_logits = torch.randn(vocab_size)
        target_probs = F.softmax(target_logits * 2, dim=0)
        
        # Draft distribution (quality depends on method)
        if method == 'small_draft':
            noise_scale = 1.0  # Independent model, moderate quality
        elif method == 'medusa':
            noise_scale = 0.7  # Shares hidden states
        elif method == 'eagle':
            noise_scale = 0.4  # Feature-level, highest quality
        elif method == 'ngram':
            noise_scale = 1.5  # Simple n-gram, lowest quality
        
        draft_logits = target_logits + torch.randn(vocab_size) * noise_scale
        draft_probs = F.softmax(draft_logits * 2, dim=0)
        
        # Expected acceptance rate = sum min(p, q)
        alpha = torch.min(target_probs, draft_probs).sum().item()
        rates.append(alpha)
    
    return rates

methods = ['ngram', 'small_draft', 'medusa', 'eagle']
method_rates = {m: simulate_acceptance_rates(m) for m in methods}

fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot([method_rates[m] for m in methods], labels=methods, patch_artist=True)
colors = ['#F44336', '#FF9800', '#2196F3', '#4CAF50']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Acceptance Rate', fontsize=12)
ax.set_title('Acceptance Rate by Speculative Decoding Method', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

# Add mean annotations
for i, m in enumerate(methods):
    mean_rate = np.mean(method_rates[m])
    ax.annotate(f'mean={mean_rate:.2f}', (i+1, mean_rate), 
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/acceptance_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Figure 2: Token Tree Speculation ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')
fig.patch.set_facecolor('white')

# Tree nodes: (x, y, label, color)
tree = [
    # Root (prompt)
    (7, 7, 'Context\n(prompt)', '#95A5A6'),
    # Level 1: two candidates
    (4.5, 5.2, '"the"', BLUE),
    (9.5, 5.2, '"a"', GREEN),
    # Level 2: two candidates per parent
    (3, 3.4, '"cat"', BLUE),
    (6, 3.4, '"dog"', BLUE),
    (8, 3.4, '"big"', GREEN),
    (11, 3.4, '"red"', GREEN),
    # Level 3: one candidate per leaf
    (2, 1.6, '"sat"', BLUE),
    (4, 1.6, '"ran"', BLUE),
    (5.5, 1.6, '"is"', BLUE),
    (7, 1.6, '"ate"', BLUE),
    (8.5, 1.6, '"cat"', GREEN),
    (10, 1.6, '"dog"', GREEN),
    (11.5, 1.6, '"hat"', GREEN),
    (13, 1.6, '"car"', GREEN),
]

bw, bh = 1.4, 0.8
for x, y, label, color in tree:
    rect = mpatches.FancyBboxPatch((x-bw/2, y-bh/2), bw, bh,
        boxstyle="round,pad=0.08", facecolor=color, alpha=0.85,
        edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=8,
            fontweight='bold', color='white')

# Edges
edges = [
    (0, 1), (0, 2),       # root -> level 1
    (1, 3), (1, 4),       # "the" -> level 2
    (2, 5), (2, 6),       # "a" -> level 2
    (3, 7), (3, 8),       # "cat" -> level 3
    (4, 9), (4, 10),      # "dog" -> level 3
    (5, 11), (5, 12),     # "big" -> level 3
    (6, 13), (6, 14),     # "red" -> level 3
]
for parent_idx, child_idx in edges:
    px, py = tree[parent_idx][0], tree[parent_idx][1]
    cx, cy = tree[child_idx][0], tree[child_idx][1]
    ax.annotate('', xy=(cx, cy+bh/2), xytext=(px, py-bh/2),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.2))

# Annotations
ax.text(7, 0.4, 'All 15 candidates verified in a SINGLE target forward pass\n'
        'using tree attention mask. Best path is selected.',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor=ORANGE))

# Highlight accepted path
accepted = [(7,7), (4.5,5.2), (3,3.4), (2,1.6)]
for i in range(len(accepted)-1):
    ax.annotate('', xy=accepted[i+1], xytext=accepted[i],
                arrowprops=dict(arrowstyle='->', color=RED, lw=3, alpha=0.6))

ax.text(0.5, 1.6, 'Accepted\npath', fontsize=9, color=RED, fontweight='bold')

ax.set_title('Token Tree Speculation: Multiple Candidate Paths Verified in Parallel',
             fontsize=14, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

## 7. Token Tree Verification

### Beyond Linear Speculation

Instead of a linear chain of draft tokens, we can build a **tree** of candidates:

```
Linear speculation (K=3):
  d1 → d2 → d3
  Total candidates: 3

Tree speculation (branching factor=2, depth=3):
         d1
        / \
      d2a  d2b
      / \   / \
    d3a d3b d3c d3d
  Total candidates: 7
  But only need to verify log(7) ≈ 3 at each level
```

### Medusa's Tree Attention

Medusa uses a tree structure where each head can produce multiple candidates:

```
Medusa tree (top-k=2 per head, 4 heads):
Head 0: [a, b] (2 candidates for t+1)
Head 1: [c, d] (2 candidates for t+2, for each parent)
Head 2: [e, f] (2 candidates for t+3)
Head 3: [g, h] (2 candidates for t+4)

All candidates verified in a SINGLE forward pass
using tree attention mask.
```

In [ ]:
def create_tree_attention_mask(tree_structure: List[List[int]]) -> torch.Tensor:
    """
    Create attention mask for tree-structured verification.
    
    tree_structure: list of lists, where tree_structure[i] contains
    the indices of position i's children.
    """
    num_nodes = len(tree_structure)
    mask = torch.zeros(num_nodes, num_nodes, dtype=torch.bool)
    
    # Each node can attend to itself and its ancestors
    def get_ancestors(node_idx):
        ancestors = {node_idx}
        # Find parent
        for parent_idx, children in enumerate(tree_structure):
            if node_idx in children:
                ancestors.update(get_ancestors(parent_idx))
                break
        return ancestors
    
    for i in range(num_nodes):
        ancestors = get_ancestors(i)
        for anc in ancestors:
            mask[i, anc] = True
    
    return mask

# Example tree:
# Node 0 (root) → children [1, 2]
# Node 1 → children [3, 4]
# Node 2 → children [5, 6]
# Nodes 3-6 → leaves
tree = [
    [1, 2],    # Node 0's children
    [3, 4],    # Node 1's children
    [5, 6],    # Node 2's children
    [],        # Leaf
    [],        # Leaf
    [],        # Leaf
    [],        # Leaf
]

mask = create_tree_attention_mask(tree)

print("Tree Attention Mask:")
print("  Rows = queries (each candidate), Cols = keys (what it can attend to)")
print(f"  Shape: {mask.shape}")
print()
for i in range(len(tree)):
    row = ['1' if mask[i, j] else '.' for j in range(len(tree))]
    print(f"  Node {i}: [{' '.join(row)}]")

print("\n  Legend: 1 = can attend, . = masked")
print("  Node 3 (leaf) can attend to: 0 (root), 1 (parent), 3 (self)")
print("  Node 5 (leaf) can attend to: 0 (root), 2 (parent), 5 (self)")

## 8. vLLM Speculative Decoding Source Code

### Architecture

```
vllm/spec_decode/
├── spec_decode_worker.py     # Main speculative decode logic
├── draft_model_runner.py     # Draft model execution
├── multi_step_worker.py      # Multi-step draft generation
├── batch_expansion.py        # Expand batch for verification
├── metrics.py                # Acceptance rate tracking
├── ngram_worker.py           # N-gram based draft (no model needed)
├── medusa_worker.py          # Medusa speculative worker
├── mlp_speculator.py         # MLP-based speculator
└── interfaces.py             # Abstract interfaces
```

### Core Flow

```python
# From vllm/spec_decode/spec_decode_worker.py (simplified)
class SpecDecodeWorker:
    def __init__(self, proposer, scorer, ...):
        self.proposer = proposer  # Draft model
        self.scorer = scorer      # Target model
        self.num_speculative_tokens = K
    
    def execute_model(self, execute_model_req):
        # Step 1: Generate draft tokens
        proposals = self.proposer.get_proposals(
            execute_model_req,
            self.num_speculative_tokens,
        )
        # proposals.token_ids: [batch, K]
        # proposals.probs: [batch, K, vocab]
        
        # Step 2: Score all proposals with target model
        # This is a SINGLE forward pass with expanded batch
        target_probs = self.scorer.score_proposals(
            execute_model_req,
            proposals,
        )
        # target_probs: [batch, K+1, vocab]
        
        # Step 3: Accept/reject
        accepted_tokens = self._verify_tokens(
            proposals.token_ids,
            proposals.probs,
            target_probs,
        )
        
        return accepted_tokens
```

In [ ]:
class SpecDecodeSimulator:
    """
    Full speculative decoding simulation with metrics tracking.
    Mirrors vllm/spec_decode/spec_decode_worker.py
    """
    
    def __init__(self, vocab_size=32000, K=5, draft_quality=0.7,
                 target_latency_ms=20.0, draft_latency_ms=2.0):
        self.vocab_size = vocab_size
        self.K = K
        self.draft_quality = draft_quality
        self.target_latency_ms = target_latency_ms
        self.draft_latency_ms = draft_latency_ms
        
        # Metrics
        self.tokens_generated = 0
        self.target_calls = 0
        self.draft_calls = 0
        self.acceptance_lengths = []  # How many tokens accepted per step
    
    def generate_tokens(self, num_tokens: int) -> dict:
        """Generate tokens using speculative decoding."""
        tokens = []
        total_time_ms = 0
        
        while len(tokens) < num_tokens:
            # Draft phase
            draft_tokens = []
            draft_probs_list = []
            for _ in range(self.K):
                target_logits = torch.randn(self.vocab_size)
                target_probs = F.softmax(target_logits * 2, dim=0)
                
                noise = torch.randn(self.vocab_size) * (1 - self.draft_quality)
                draft_logits = target_logits + noise
                draft_probs = F.softmax(draft_logits * 2, dim=0)
                
                draft_token = torch.multinomial(draft_probs, 1).item()
                draft_tokens.append(draft_token)
                draft_probs_list.append((target_probs, draft_probs))
            
            self.draft_calls += self.K
            total_time_ms += self.K * self.draft_latency_ms
            
            # Verify phase (single target call)
            self.target_calls += 1
            total_time_ms += self.target_latency_ms
            
            # Accept/reject
            accepted = 0
            for k in range(self.K):
                target_probs, draft_probs = draft_probs_list[k]
                _, was_accepted = speculative_sampling(
                    target_probs, draft_probs, draft_tokens[k])
                
                if was_accepted:
                    accepted += 1
                    tokens.append(draft_tokens[k])
                else:
                    tokens.append(draft_tokens[k])  # Resampled token
                    accepted += 1
                    break
            else:
                # All accepted, bonus token
                tokens.append(0)  # Bonus token
                accepted += 1
            
            self.acceptance_lengths.append(accepted)
            self.tokens_generated += accepted
        
        return {
            'tokens_generated': len(tokens),
            'total_time_ms': total_time_ms,
            'tokens_per_second': len(tokens) / (total_time_ms / 1000),
            'avg_accepted_per_step': np.mean(self.acceptance_lengths),
            'target_calls': self.target_calls,
            'draft_calls': self.draft_calls,
        }

# Compare with standard decoding
num_tokens = 200
target_latency = 20.0  # ms

# Standard decoding
standard_time = num_tokens * target_latency
standard_tps = num_tokens / (standard_time / 1000)

print(f"{'Method':<25} {'Time (ms)':>10} {'Tok/s':>8} {'Speedup':>8}")
print("-" * 55)
print(f"{'Standard decoding':<25} {standard_time:>10.0f} {standard_tps:>8.1f} {'1.0x':>8}")

for quality, name in [(0.5, 'Weak draft'), (0.7, 'Medium draft'), (0.9, 'Strong draft')]:
    sim = SpecDecodeSimulator(
        K=5, draft_quality=quality,
        target_latency_ms=target_latency, draft_latency_ms=2.0)
    result = sim.generate_tokens(num_tokens)
    speedup = result['tokens_per_second'] / standard_tps
    print(f"{name + f' (q={quality})':<25} {result['total_time_ms']:>10.0f} "
          f"{result['tokens_per_second']:>8.1f} {speedup:>7.2f}x")

## 9. When Does Speculation Help?

### Factors Affecting Speedup

| Factor | Helps | Hurts |
|--------|-------|-------|
| Draft quality | High agreement with target | Low agreement → many rejections |
| Draft cost | Fast draft (tiny model) | Expensive draft |
| Batch size | Small batches (memory-bound decode) | Large batches (compute-bound) |
| Sequence type | Predictable text (code, data) | Creative text (stories) |
| Temperature | Low temp (sharp distributions) | High temp (flat distributions) |

In [ ]:
# Sensitivity analysis

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Speedup vs draft quality
qualities = np.linspace(0.1, 0.99, 30)
for K in [3, 5, 7]:
    speedups = [speculative_speedup(q, K, draft_cost=0.05) for q in qualities]
    axes[0].plot(qualities, speedups, label=f'K={K}', linewidth=2)
axes[0].axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Draft Quality (acceptance rate)', fontsize=11)
axes[0].set_ylabel('Speedup', fontsize=11)
axes[0].set_title('Speedup vs Draft Quality', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Speedup vs draft cost
draft_costs = np.linspace(0.01, 0.5, 30)
for alpha in [0.6, 0.75, 0.9]:
    speedups = [speculative_speedup(alpha, 5, dc) for dc in draft_costs]
    axes[1].plot(draft_costs, speedups, label=f'alpha={alpha}', linewidth=2)
axes[1].axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Draft Cost (relative to target)', fontsize=11)
axes[1].set_ylabel('Speedup', fontsize=11)
axes[1].set_title('Speedup vs Draft Cost (K=5)', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Speedup vs K
K_range = range(1, 20)
for alpha in [0.6, 0.75, 0.9]:
    speedups = [speculative_speedup(alpha, k, 0.05) for k in K_range]
    axes[2].plot(list(K_range), speedups, 'o-', label=f'alpha={alpha}', linewidth=2, markersize=4)
axes[2].axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
axes[2].set_xlabel('K (draft tokens)', fontsize=11)
axes[2].set_ylabel('Speedup', fontsize=11)
axes[2].set_title('Speedup vs K', fontsize=13)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/spec_decode_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Configuration in vLLM

```python
# Draft model approach:
vllm serve meta-llama/Llama-2-70b-chat-hf \
    --speculative-model meta-llama/Llama-2-7b-chat-hf \
    --num-speculative-tokens 5 \
    --speculative-draft-tensor-parallel-size 1

# N-gram based (no draft model needed):
vllm serve meta-llama/Llama-2-70b-chat-hf \
    --speculative-model [ngram] \
    --num-speculative-tokens 5 \
    --ngram-prompt-lookup-max 4

# Medusa:
vllm serve meta-llama/Llama-2-70b-chat-hf \
    --speculative-model path/to/medusa/heads \
    --spec-decoding-method medusa

# Python API:
from vllm import LLM
llm = LLM(
    model="meta-llama/Llama-2-70b-chat-hf",
    speculative_model="meta-llama/Llama-2-7b-chat-hf",
    num_speculative_tokens=5,
)
```

## 11. Summary

| Method | Draft Source | Acceptance Rate | Extra Memory | Complexity |
|--------|-------------|----------------|-------------|------------|
| Draft Model | Separate small LLM | Medium-High | Model weights | Medium |
| Medusa | Extra heads on target | Medium | Small (MLP heads) | Low |
| Eagle | Feature-level head | High | Small | Medium |
| MLPSpeculator | MLP on hidden states | Medium | Very small | Low |
| N-gram | Prompt n-gram lookup | Low-Medium | None | Very low |

### Typical Speedups
- **Draft model** (good quality): 1.5-2.5x
- **Medusa**: 1.5-2.0x
- **Eagle**: 2.0-3.0x
- **N-gram**: 1.2-1.5x (for repetitive text)

## Exercises

### Exercise 1: Optimal K Selection
Given a draft model with acceptance rate alpha=0.8 and cost ratio 0.1, find the optimal K that maximizes throughput.

### Exercise 2: Adaptive Speculation
Design a system that dynamically adjusts K based on recent acceptance rates. Implement and compare with fixed K.

### Exercise 3: Batch Speculation
Analyze how speculative decoding works with batched requests. What happens when different sequences have different acceptance rates?

### Exercise 4: N-gram Speculator
Implement an n-gram based speculator that looks up candidate tokens from the prompt itself. Test on code completion tasks.

In [ ]:
# Exercise 1: Find optimal K
alpha = 0.8
draft_cost = 0.1

K_range = range(1, 30)
speedups = [speculative_speedup(alpha, k, draft_cost) for k in K_range]

optimal_K = K_range[np.argmax(speedups)]
max_speedup = max(speedups)

print(f"Acceptance rate: {alpha}")
print(f"Draft cost ratio: {draft_cost}")
print(f"Optimal K: {optimal_K}")
print(f"Maximum speedup: {max_speedup:.2f}x")

plt.figure(figsize=(10, 5))
plt.plot(list(K_range), speedups, 'bo-', markersize=4)
plt.axvline(x=optimal_K, color='r', linestyle='--', label=f'Optimal K={optimal_K}')
plt.xlabel('K (draft tokens)')
plt.ylabel('Speedup')
plt.title(f'Finding Optimal K (alpha={alpha}, draft_cost={draft_cost})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 4 Starter: N-gram speculator

class NgramSpeculator:
    """
    Simple n-gram based speculator that looks up candidate tokens
    from patterns in the prompt.
    
    Mirrors: vllm/spec_decode/ngram_worker.py
    """
    
    def __init__(self, n: int = 3, max_speculation: int = 5):
        self.n = n
        self.max_speculation = max_speculation
    
    def get_proposals(self, token_ids: List[int]) -> List[int]:
        """
        Look for n-gram matches in the prompt and predict next tokens.
        """
        if len(token_ids) < self.n:
            return []
        
        # Get the last n tokens as the query pattern
        query = tuple(token_ids[-self.n:])
        proposals = []
        
        # Search for this pattern earlier in the sequence
        for i in range(len(token_ids) - self.n):
            window = tuple(token_ids[i:i + self.n])
            if window == query and i + self.n < len(token_ids):
                # Found a match! Use following tokens as proposals
                for j in range(self.max_speculation):
                    idx = i + self.n + j
                    if idx < len(token_ids):
                        proposals.append(token_ids[idx])
                    else:
                        break
                return proposals
        
        return []  # No match found

# Test with repetitive code
spec = NgramSpeculator(n=3, max_speculation=5)

# Simulated token sequence (repetitive pattern in code)
# pattern: [10, 20, 30, 40, 50] repeats
tokens = [10, 20, 30, 40, 50, 10, 20, 30, 40, 50, 10, 20, 30]

proposals = spec.get_proposals(tokens)
print(f"Input tokens: {tokens}")
print(f"Last 3-gram: {tokens[-3:]}")
print(f"Proposals: {proposals}")
print(f"Expected next: {[40, 50]} (from the repeating pattern)")